In [1]:
from pathlib import Path
import pandas as pd

project_dir = Path(r"C:\Users\HP-ZBOOK i7\graphrag_test\pilot_01")
csv_path = project_dir / "inspection" / "08_documents_all.csv"

documents = pd.read_csv(csv_path)

print("Rows:", len(documents))
print("Columns:", documents.columns.tolist())
documents

Rows: 1
Columns: ['id', 'human_readable_id', 'title', 'text', 'text_unit_ids', 'creation_date', 'raw_data']


,id,human_readable_id,title,text,text_unit_ids,creation_date,raw_data
0,39ed356c790f07e222ffe8e84bfb78009537e51e126410...,0,AT_STRATEGY_Austrian_Photovoltaic_Strategy_202...,Österreichische Photovoltaik-\nStrategie\n\nZi...,['d7219f76b31818b7b13bf08dda81003d752e53f76608...,2026-07-26 22:20:50 +0200,NaN


In [2]:
document_text = documents.loc[0, "text"]

print("Title:", documents.loc[0, "title"])
print("Number of characters:", len(document_text))
print()
print(document_text[:5000])

Title: AT_STRATEGY_Austrian_Photovoltaic_Strategy_2024_DE.pdf
Number of characters: 73134

Österreichische Photovoltaik-
Strategie

Zielsetzungen und Aktionsfelder eines strategischen Ausbauprozesses

sowie Maßnahmen für einen koordinierten Ausbau der Photovoltaik

in Österreich

Impressum

Medieninhaber, Verleger und Herausgeber:

Bundesministerium für Klimaschutz, Umwelt, Energie, Mobilität,

Innovation und Technologie, Radetzkystraße 2, 1030 Wien

Autor: Hubert Fechner

Fotonachweis: stock.adobe.com – Alan (Cover), BMK/Cajetan Perwein (Vorwort)

Wien, 2024.

Vorwort

Die Klimakrise ist eine der größten Herausforderung unserer Zeit.

Wir spüren ihre Auswirkungen immer deutlicher. Als österreichi-

sche  Bundesregierung haben  wir uns  daher  ein ehrgeiziges  Ziel

gesetzt: ein klimaneutrales Österreich bis 2040. Zur Bekämpfung

der Klimakrise sind viele Maßnahmen notwendig, insbesondere

ein Vorantreiben der Energiewende.

Leonore Gewessler

Unser Energiesystem klimaverträglich, fl

In [3]:
important_terms = [
    "Photovoltaik",
    "Klimaneutralität",
    "Erneuerbaren-Ausbau-Gesetz",
    "41 TWh",
    "2040",
    "2,5 GW",
    "ÖNIP",
]

for term in important_terms:
    print(term, ":", document_text.count(term))

print()
print("Page breaks:", document_text.count("\f"))
print("Hyphen plus space:", document_text.count("- "))
print("Multiple spaces:", document_text.count("  "))

Photovoltaik : 102
Klimaneutralität : 6
Erneuerbaren-Ausbau-Gesetz : 7
41 TWh : 4
2040 : 26
2,5 GW : 2
ÖNIP : 1

Page breaks: 43
Hyphen plus space: 74
Multiple spaces: 999


In [4]:
import re

hyphen_examples = re.findall(
    r"\b[\wÄÖÜäöüß]+-\s+[\wÄÖÜäöüß]+\b",
    document_text
)

print("Potential split-word examples:", len(hyphen_examples))
print()

for example in hyphen_examples[:30]:
    print(example)

Potential split-word examples: 333

Photovoltaik-
Strategie
österreichi-

sche
weiter-

zuentwickeln
Herausfor-

derungen
Energie-

unabhängigkeit
Stromversor-

gung
natio-

nalen
Klima- und
ins-

besondere
heuti-

gen
zuge-

baut
österreichi-

schen
Maß-

nahmen
Energie- und
Bünde-

lung
be-

sonders
Ge-

meinsam
Ener-

giesystems
Ener-

giequellen
mit-

gestaltet
Energieversor-

gung
konstruk-

tives
Landes- und
Gemein-

deebene
Energieinfra-

struktur
Bevölke-

rung
Wirt-

schaft
de-

ren
ge-

tretenen
euro-

päische


## Stage 1 — PDF-to-Text Conversion Audit

### Objective

The purpose of this stage was to assess whether the original Austrian Photovoltaic Strategy PDF was converted into sufficiently accurate text before GraphRAG performed chunking, entity extraction, and relationship extraction.

The original PDF was treated as the authoritative source. The converted text was obtained from `08_documents_all.csv`, representing the text received by GraphRAG after MarkItDown processing.

### Basic results

* Source documents processed: **1**
* Original PDF length: **44 pages**
* Converted-text length: **73,134 characters**
* Detected page-break characters: **43**

The 43 page-break characters are consistent with a 44-page document, suggesting that all pages were represented in the converted text.

### Preservation of important information

Several important terms, abbreviations, targets, and numerical values were found in the converted text:

| Term                       | Exact occurrences |
| -------------------------- | ----------------: |
| Photovoltaik               |               102 |
| Klimaneutralität           |                 6 |
| Erneuerbaren-Ausbau-Gesetz |                 7 |
| 41 TWh                     |                 4 |
| 2040                       |                26 |
| 2,5 GW                     |                 2 |
| ÖNIP                       |                 1 |

These results indicate that important policy names, abbreviations, years, units, and numerical targets generally survived the conversion. German characters such as `ä`, `ö`, `ü`, `Ö`, and `ß` were also preserved.

### Identified conversion problems

The converted text contains substantial PDF-layout noise.

#### Broken words caused by line-end hyphenation

A regular-expression screening identified **333 potential hyphenation cases**. Many represent words that were incorrectly divided because of PDF line wrapping.

Examples include:

| Converted text            | Intended text           |
| ------------------------- | ----------------------- |
| `österreichi- sche`       | `österreichische`       |
| `weiter- zuentwickeln`    | `weiterzuentwickeln`    |
| `Herausfor- derungen`     | `Herausforderungen`     |
| `Energie- unabhängigkeit` | `Energieunabhängigkeit` |
| `Stromversor- gung`       | `Stromversorgung`       |
| `natio- nalen`            | `nationalen`            |
| `zuge- baut`              | `zugebaut`              |
| `Maß- nahmen`             | `Maßnahmen`             |

However, not all detected cases are errors. Some German coordinated compounds legitimately retain the hyphen and following space, for example:

* `Klima- und Energieziele`
* `Landes- und Gemeindeebene`

A third case occurs when the hyphen is correct but the following space is not:

* `Photovoltaik- Strategie` should become `Photovoltaik-Strategie`.

Consequently, a global replacement that removes every hyphen followed by whitespace would be unsafe and could damage valid German text.

#### Additional layout noise

The converted text also contains:

* Repeated page headers and footers
* Page numbers such as `3 von 44`
* Table-of-contents formatting characters
* Large amounts of repeated whitespace
* Form-feed characters representing page boundaries
* Occasional imperfect reading order caused by multi-column or image-and-text layouts

The diagnostic found **999 occurrences of consecutive spaces**. This number is only an approximate indicator of formatting noise and should not be interpreted as 999 substantive conversion errors.

### Preliminary assessment

MarkItDown preserved most of the document’s substantive content, including major policy concepts, numerical targets, abbreviations, German characters, and page boundaries. The conversion was therefore not a complete failure.

Nevertheless, the large number of potential word-splitting artifacts and the retained layout elements show that the text was not fully cleaned before chunking. These problems may negatively affect:

* Exact keyword matching
* Text embeddings
* Entity identification
* Entity resolution
* Duplicate detection
* Relationship extraction
* Source interpretation at chunk boundaries

### Stage 1 conclusion

**PDF conversion is assessed as a meaningful but manageable preprocessing bottleneck.** The substantive content is mostly preserved, but the converted text contains enough hyphenation and layout noise to potentially influence downstream GraphRAG extraction quality.

No preprocessing was applied at this stage because the current output must be preserved as the baseline for Experiment 1. After completing the full pipeline audit, a separate Experiment 2 may test careful preprocessing, including:

* Context-aware dehyphenation
* Whitespace normalization
* Removal of repeated headers and footers
* Preservation of headings and page boundaries
* Improved treatment of tables and layout elements

Any cleaned input should be stored separately and compared against the unchanged Experiment 1 baseline.
